# Phase 6 — Comparative Analysis

This notebook consolidates the results from the traditional machine learning and graph-based modeling phases.

The purpose of this phase is to compare model performance under the leakage-safe temporal evaluation framework defined in Phase 3. The analysis focuses on three components:

1. comparing traditional machine learning models with graph-based approaches,
2. assessing the contribution of leakage-safe structural interaction features,
3. evaluating whether LightGCN-derived user and streamer representations provide additional predictive value beyond engineered behavioral features.

The comparison is conducted carefully because not all models are evaluated on the exact same test universe. Traditional models are evaluated on the full temporal test set, while LightGCN-based models can only be evaluated on anchors where both the user and streamer are present in the training graph. This distinction is explicitly documented to avoid overstating comparability.

The final outputs of this notebook are thesis-ready comparison tables, model interpretation summaries, and research-question-level conclusions.

In [48]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Evaluation metrics
from sklearn.metrics import (
    auc,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project paths
DATA_DIR = Path("../data_processed")
REPORTS_DIR = Path("../reports/modeling")
TABLE_DIR = REPORTS_DIR / "tables"
FIGURE_DIR = REPORTS_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Input files
FEATURE_DATA_PATH = DATA_DIR / "model_dataset_100k_3day_features_v1.csv"
TRADITIONAL_RESULTS_PATH = TABLE_DIR / "traditional_models_results.csv"

print("Setup complete.")
print("Feature dataset exists:", FEATURE_DATA_PATH.exists())
print("Traditional results file exists:", TRADITIONAL_RESULTS_PATH.exists())

Setup complete.
Feature dataset exists: True
Traditional results file exists: True


In [23]:
# Load full feature dataset
df = pd.read_csv(FEATURE_DATA_PATH)

# Full temporal splits
train_full = df[df["segment"] == "train"].copy()
val_full = df[df["segment"] == "validation"].copy()
test_full = df[df["segment"] == "test"].copy()

print("Full train shape:", train_full.shape)
print("Full validation shape:", val_full.shape)
print("Full test shape:", test_full.shape)

Full train shape: (1373160, 30)
Full validation shape: (520761, 30)
Full test shape: (486650, 30)


## Graph-Covered Evaluation Subset Analysis

LightGCN-based models can only generate predictions for users and streamers that are present in the training interaction graph.

As a result, graph-based evaluation is performed on a filtered subset of the original validation and test sets. Before comparing graph-based models with traditional baselines, the class distribution of these filtered subsets is inspected.

This analysis helps determine whether the graph-covered evaluation subset remains representative of the original temporal splits or introduces substantial distributional bias.

In [24]:
# Graph-covered subsets from Phase 5B

graph_covered_summary = pd.DataFrame([
    {
        "split": "validation_full",
        "rows": len(val_full),
        "positive_rate": val_full["label"].mean()
    },
    {
        "split": "validation_graph_covered",
        "rows": 459847,
        "positive_rate": 115316 / 459847
    },
    {
        "split": "test_full",
        "rows": len(test_full),
        "positive_rate": test_full["label"].mean()
    },
    {
        "split": "test_graph_covered",
        "rows": 403830,
        "positive_rate": 96813 / 403830
    }
])

graph_covered_summary["positive_rate"] = (
    graph_covered_summary["positive_rate"]
    .round(4)
)

display(graph_covered_summary)

,split,rows,positive_rate
0,validation_full,520761,0.2398
1,validation_graph_covered,459847,0.2508
2,test_full,486650,0.2397
3,test_graph_covered,403830,0.2397


### Interpretation of Graph-Covered Evaluation Subsets

The graph-covered validation and test subsets remain broadly consistent with the original temporal splits in terms of class distribution.

Most importantly, the positive label rate in the graph-covered test subset (23.97%) is effectively identical to the positive rate of the full temporal test set (23.97%). This indicates that restricting evaluation to anchors covered by the training interaction graph does not materially distort the class balance of the final evaluation data.

A slightly higher positive rate is observed in the graph-covered validation subset (25.08% compared to 23.98% in the full validation set). However, this difference remains relatively small and does not substantially affect the role of the validation set for threshold selection and model comparison.

Overall, these results suggest that the filtered graph-compatible evaluation subsets remain broadly representative of the original temporal splits, supporting the methodological validity of the LightGCN evaluation setup despite its transductive limitations.

## Re-Evaluation of Traditional Models on the Graph-Covered Test Subset

The graph-based models are evaluated only on anchors where both the user and streamer are present in the training interaction graph.

To ensure a fair comparison, the traditional machine learning baselines are re-evaluated on the exact same graph-covered test subset.

This step isolates the effect of the feature representation from potential differences in evaluation coverage and allows a direct comparison between:

- tabular-only models,
- graph-only representations,
- hybrid tabular + graph models.

No retraining is performed. The original trained models and validation-selected thresholds from Phase 5A are reused.

In [25]:
# Load graph-covered test subset identifiers from Phase 5B

GRAPH_TEST_KEYS = ["user_id", "streamer_name", "prediction_time"]

# Reconstruct graph-covered test subset
# using the same coverage condition as Phase 5B

train_users = set(train_full["user_id"].unique())
train_streamers = set(train_full["streamer_name"].unique())

graph_test_mask = (
    test_full["user_id"].isin(train_users) &
    test_full["streamer_name"].isin(train_streamers)
)

test_graph_covered = test_full[graph_test_mask].copy()

print("Full test rows:", len(test_full))
print("Graph-covered test rows:", len(test_graph_covered))
print("Coverage rate:", round(len(test_graph_covered) / len(test_full), 4))

Full test rows: 486650
Graph-covered test rows: 403830
Coverage rate: 0.8298


### Reconstruction of Traditional Baselines

The original Phase 5A traditional models are reconstructed using the same:

- feature set,
- training split,
- preprocessing pipeline,
- hyperparameters,
- threshold-selection strategy.

The purpose of this step is not to improve performance, but to obtain directly comparable results on the graph-covered evaluation subset used by the LightGCN-based models.

In [26]:
# Final Phase 4 feature set

FEATURE_COLUMNS = [

    # User-level
    "user_past_sessions",
    "user_total_watch_time",
    "user_avg_session_duration",
    "user_recency",
    "user_has_history",

    # Streamer-level
    "streamer_past_sessions",
    "streamer_total_watch_time",
    "streamer_avg_session_duration",
    "streamer_recency",
    "streamer_has_history",

    # Pair-level
    "pair_past_sessions",
    "pair_total_watch_time",
    "pair_avg_session_duration",
    "pair_recency",
    "pair_has_history",

    # Temporal
    "time_sin_day",
    "time_cos_day",
    "time_sin_week",
    "time_cos_week",

    # Structural / behavioral
    "user_unique_streamers",
    "streamer_unique_users",
    "user_repeat_ratio",
    "streamer_repeat_audience_ratio",
    "pair_watch_share_user",
    "pair_watch_share_streamer",
]

TARGET = "label"

In [27]:
# Prepare train and graph-covered test matrices

X_train = train_full[FEATURE_COLUMNS]
y_train = train_full[TARGET]

X_test_graph = test_graph_covered[FEATURE_COLUMNS]
y_test_graph = test_graph_covered[TARGET]

print("Train shape:", X_train.shape)
print("Graph-covered test shape:", X_test_graph.shape)

Train shape: (1373160, 25)
Graph-covered test shape: (403830, 25)


In [28]:
# Reconstruct Logistic Regression baseline from Phase 5A

lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)

print("Logistic Regression reconstruction complete.")

Logistic Regression reconstruction complete.


In [29]:
def evaluate_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

In [30]:
# Phase 5A selected threshold
LR_THRESHOLD = 0.26

# Predict on graph-covered test subset
lr_graph_subset_probs = lr_pipeline.predict_proba(X_test_graph)[:, 1]

lr_graph_subset_results = evaluate_threshold(
    y_test_graph,
    lr_graph_subset_probs,
    LR_THRESHOLD
)

print("Logistic Regression — Graph-Covered Test Subset")
for k, v in lr_graph_subset_results.items():
    print(f"{k}: {v:.4f}")

Logistic Regression — Graph-Covered Test Subset
threshold: 0.2600
roc_auc: 0.7038
pr_auc: 0.4534
f1: 0.4909
precision: 0.3782
recall: 0.6991


In [31]:
# Reconstruct tuned XGBoost baseline from Phase 5A

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("model", XGBClassifier(
        max_depth=4,
        learning_rate=0.05,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.0,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)

print("Tuned XGBoost reconstruction complete.")

Tuned XGBoost reconstruction complete.


In [32]:
# Phase 5A selected threshold
XGB_THRESHOLD = 0.25

# Predict on graph-covered test subset
xgb_graph_subset_probs = xgb_pipeline.predict_proba(X_test_graph)[:, 1]

xgb_graph_subset_results = evaluate_threshold(
    y_test_graph,
    xgb_graph_subset_probs,
    XGB_THRESHOLD
)

print("Tuned XGBoost — Graph-Covered Test Subset")
for k, v in xgb_graph_subset_results.items():
    print(f"{k}: {v:.4f}")

Tuned XGBoost — Graph-Covered Test Subset
threshold: 0.2500
roc_auc: 0.7045
pr_auc: 0.4525
f1: 0.4881
precision: 0.3672
recall: 0.7280


## Final Model Comparison — Full Temporal Test Set

This table summarizes the final performance of the traditional machine learning baselines evaluated on the complete temporal test set.

All models use the same leakage-safe temporal split and validation-based threshold selection procedure established in Phase 5A.

In [33]:
# Full temporal test set comparison table

full_test_results = pd.DataFrame([

    {
        "model": "Majority Class Baseline",
        "evaluation_scope": "full_test",
        "roc_auc": 0.5000,
        "pr_auc": np.nan,
        "f1": 0.0000,
        "precision": 0.0000,
        "recall": 0.0000,
        "threshold": "-"
    },

    {
        "model": "Logistic Regression",
        "evaluation_scope": "full_test",
        "roc_auc": 0.7054,
        "pr_auc": 0.4403,
        "f1": 0.4800,
        "precision": 0.3776,
        "recall": 0.6586,
        "threshold": 0.26
    },

    {
        "model": "Tuned XGBoost",
        "evaluation_scope": "full_test",
        "roc_auc": 0.7064,
        "pr_auc": 0.4395,
        "f1": 0.4779,
        "precision": 0.3678,
        "recall": 0.6822,
        "threshold": 0.25
    }

])

display(full_test_results)

,model,evaluation_scope,roc_auc,pr_auc,f1,precision,recall,threshold
0,Majority Class Baseline,full_test,0.5000,NaN,0.0000,0.0000,0.0000,-
1,Logistic Regression,full_test,0.7054,0.4403,0.4800,0.3776,0.6586,0.26
2,Tuned XGBoost,full_test,0.7064,0.4395,0.4779,0.3678,0.6822,0.25


In [34]:
# Save full test comparison table

FULL_RESULTS_PATH = TABLE_DIR / "full_test_comparison.csv"

full_test_results.to_csv(FULL_RESULTS_PATH, index=False)

print("Saved to:", FULL_RESULTS_PATH)

Saved to: ../reports/modeling/tables/full_test_comparison.csv


## Final Model Comparison — Graph-Covered Test Subset

This table presents the direct comparison between traditional, graph-only, and hybrid models evaluated on the same graph-covered test subset.

Unlike the full temporal test evaluation, this comparison controls for graph coverage by restricting all models to anchors where both the user and streamer are present in the training interaction graph.

This enables a fair assessment of whether LightGCN-derived graph representations provide additional predictive value beyond engineered tabular features.

In [35]:
# Graph-covered subset comparison table

graph_subset_results = pd.DataFrame([

    {
        "model": "Logistic Regression (tabular)",
        "feature_type": "tabular_only",
        "evaluation_scope": "graph_covered_test",
        "roc_auc": round(lr_graph_subset_results["roc_auc"], 4),
        "pr_auc": round(lr_graph_subset_results["pr_auc"], 4),
        "f1": round(lr_graph_subset_results["f1"], 4),
        "precision": round(lr_graph_subset_results["precision"], 4),
        "recall": round(lr_graph_subset_results["recall"], 4),
        "threshold": LR_THRESHOLD
    },

    {
        "model": "Tuned XGBoost (tabular)",
        "feature_type": "tabular_only",
        "evaluation_scope": "graph_covered_test",
        "roc_auc": round(xgb_graph_subset_results["roc_auc"], 4),
        "pr_auc": round(xgb_graph_subset_results["pr_auc"], 4),
        "f1": round(xgb_graph_subset_results["f1"], 4),
        "precision": round(xgb_graph_subset_results["precision"], 4),
        "recall": round(xgb_graph_subset_results["recall"], 4),
        "threshold": XGB_THRESHOLD
    },

    {
        "model": "LightGCN + Logistic Regression",
        "feature_type": "graph_only",
        "evaluation_scope": "graph_covered_test",
        "roc_auc": 0.5801,
        "pr_auc": 0.3061,
        "f1": 0.4069,
        "precision": 0.2696,
        "recall": 0.8290,
        "threshold": 0.18
    },

    {
        "model": "Hybrid XGBoost",
        "feature_type": "tabular_plus_graph",
        "evaluation_scope": "graph_covered_test",
        "roc_auc": 0.6978,
        "pr_auc": 0.4512,
        "f1": 0.4847,
        "precision": 0.3770,
        "recall": 0.6780,
        "threshold": 0.23
    }

])

display(graph_subset_results)

,model,feature_type,evaluation_scope,roc_auc,pr_auc,f1,precision,recall,threshold
0,Logistic Regression (tabular),tabular_only,graph_covered_test,0.7038,0.4534,0.4909,0.3782,0.6991,0.26
1,Tuned XGBoost (tabular),tabular_only,graph_covered_test,0.7045,0.4525,0.4881,0.3672,0.7280,0.25
2,LightGCN + Logistic Regression,graph_only,graph_covered_test,0.5801,0.3061,0.4069,0.2696,0.8290,0.18
3,Hybrid XGBoost,tabular_plus_graph,graph_covered_test,0.6978,0.4512,0.4847,0.3770,0.6780,0.23


In [36]:
# Save graph-covered comparison table

GRAPH_RESULTS_PATH = TABLE_DIR / "graph_subset_comparison.csv"

graph_subset_results.to_csv(GRAPH_RESULTS_PATH, index=False)

print("Saved to:", GRAPH_RESULTS_PATH)

Saved to: ../reports/modeling/tables/graph_subset_comparison.csv


## Comparative Interpretation of Model Performance

The final experimental results reveal a clear performance difference between the traditional machine learning models and the graph-based approaches.

Across the full temporal test set, Logistic Regression and tuned XGBoost achieve highly similar performance, both reaching a ROC-AUC of approximately 0.70 and F1-scores close to 0.48. This indicates that the engineered behavioral, temporal, and structural interaction features already capture substantial predictive signal for the re-engagement task.

The graph-only LightGCN representation performs substantially worse than the tabular baselines. When the LightGCN embeddings are used as the sole input to Logistic Regression, ROC-AUC decreases to approximately 0.58, indicating limited ranking capability. Diagnostic analysis further revealed severe score saturation, where predicted probabilities concentrated near 1.0 for both classes, reducing class separability.

Most importantly, the hybrid model combining tabular features with LightGCN-derived embeddings does not outperform the tabular-only XGBoost baseline when evaluated on the same graph-covered test subset. In fact, the hybrid model achieves slightly lower ROC-AUC and F1-score than the tabular-only baseline.

This suggests that, under the current static leakage-safe graph construction and training configuration, the LightGCN embeddings do not provide substantial additional predictive value beyond the engineered behavioral and interaction features already available in the tabular representation.

These findings do not imply that graph-based approaches are inherently ineffective for re-engagement prediction. Rather, they indicate that the specific static LightGCN configuration used in this study is insufficient to outperform carefully engineered leakage-safe tabular features. More advanced graph formulations, temporal interaction modeling, weighted edges, or inductive architectures may be required to better capture the dynamics of user re-engagement behavior.

## Research Question-Oriented Interpretation

### Main Research Question

**How effectively can user re-engagement on live streaming platforms be predicted using traditional machine learning models compared to graph-based approaches under a leakage-safe temporal evaluation setting?**

The results show that user re-engagement can be predicted with moderate accuracy under a strict leakage-safe temporal evaluation framework. Traditional machine learning models achieved stable performance with ROC-AUC values around 0.70 and F1-scores close to 0.48.

In contrast, the graph-based LightGCN approach produced substantially weaker predictive performance in its graph-only configuration. Furthermore, incorporating LightGCN-derived embeddings into a hybrid model did not improve performance beyond the engineered tabular feature baseline.

Overall, the findings suggest that traditional machine learning models using carefully engineered behavioral and interaction features outperform the static graph-based approach implemented in this study.

---

### SQ1

**How does the predictive performance of traditional machine learning models compare to graph-based approaches when predicting whether a user will re-engage with a streamer?**

Traditional machine learning models consistently outperformed the graph-based LightGCN approach across all evaluation metrics. Logistic Regression and tuned XGBoost achieved substantially higher ROC-AUC and F1-scores than the graph-only LightGCN representation.

Additionally, the hybrid graph-enhanced model failed to improve over the tabular-only XGBoost baseline when evaluated on the same graph-covered test subset. This indicates that the graph-based representations did not provide sufficient additional predictive signal under the implemented setup.

---

### SQ2

**To what extent do leakage-safe structural interaction features contribute to predicting user re-engagement on live streaming platforms?**

The strong performance of the traditional machine learning models suggests that the engineered structural and interaction-based features contribute substantial predictive value to the re-engagement task.

Features capturing interaction frequency, historical engagement intensity, repeated interaction behavior, and user–streamer structural relationships appear sufficient to capture much of the predictive signal available in the dataset. The limited gains from graph embeddings further reinforce the importance of these carefully engineered leakage-safe structural features.

---

### SQ3

**To what extent do LightGCN-derived user and streamer representations provide predictive value beyond engineered behavioural features in a temporal re-engagement setting?**

The LightGCN-derived embeddings provided limited additional predictive value beyond the engineered behavioral and structural features already present in the tabular representation.

Although the hybrid model performed comparably in F1, it did not match the tabular-only XGBoost baseline in ROC-AUC. This suggests that the learned graph embeddings did not capture substantial new information beyond what was already encoded in the engineered features.

One likely explanation is that the static transductive LightGCN setup was not well aligned with the temporal dynamics of the re-engagement prediction task. In particular, the graph construction did not explicitly model temporal edge evolution or interaction intensity over time.

## Structural Feature Contribution Analysis

To better understand the contribution of leakage-safe structural interaction features, an additional ablation-style analysis is performed using the tuned XGBoost model.

The goal is to evaluate whether structural interaction features provide measurable predictive value beyond the behavioral, temporal, and interaction-history features already present in the dataset.

Three feature configurations are compared:

1. behavioral + temporal features only,
2. structural interaction features only,
3. the full feature set combining all feature groups.

This analysis directly supports SQ2 by quantifying the predictive contribution of the engineered structural interaction features.

In [38]:
# Feature group definitions

BEHAVIORAL_TEMPORAL_FEATURES = [

    # User-level
    "user_past_sessions",
    "user_total_watch_time",
    "user_avg_session_duration",
    "user_recency",
    "user_has_history",

    # Streamer-level
    "streamer_past_sessions",
    "streamer_total_watch_time",
    "streamer_avg_session_duration",
    "streamer_recency",
    "streamer_has_history",

    # Pair-level
    "pair_past_sessions",
    "pair_total_watch_time",
    "pair_avg_session_duration",
    "pair_recency",
    "pair_has_history",

    # Temporal
    "time_sin_day",
    "time_cos_day",
    "time_sin_week",
    "time_cos_week",
]

STRUCTURAL_FEATURES = [
    "user_unique_streamers",
    "streamer_unique_users",
    "user_repeat_ratio",
    "streamer_repeat_audience_ratio",
    "pair_watch_share_user",
    "pair_watch_share_streamer",
]

FULL_FEATURE_SET = (
    BEHAVIORAL_TEMPORAL_FEATURES +
    STRUCTURAL_FEATURES
)

print("Behavioral + temporal features:", len(BEHAVIORAL_TEMPORAL_FEATURES))
print("Structural features:", len(STRUCTURAL_FEATURES))
print("Full feature set:", len(FULL_FEATURE_SET))

Behavioral + temporal features: 19
Structural features: 6
Full feature set: 25


In [39]:
def run_xgb_feature_group_experiment(
    train_df,
    test_df,
    feature_columns,
    threshold=0.25
):
    """
    Trains and evaluates the tuned XGBoost configuration
    on a selected feature group.
    """

    X_train_group = train_df[feature_columns]
    y_train_group = train_df["label"]

    X_test_group = test_df[feature_columns]
    y_test_group = test_df["label"]

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
        ("model", XGBClassifier(
            max_depth=4,
            learning_rate=0.05,
            n_estimators=200,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1.0,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train_group, y_train_group)

    probs = pipeline.predict_proba(X_test_group)[:, 1]

    results = evaluate_threshold(
        y_test_group,
        probs,
        threshold
    )

    return results

In [40]:
# Structural contribution experiments

behavioral_results = run_xgb_feature_group_experiment(
    train_full,
    test_full,
    BEHAVIORAL_TEMPORAL_FEATURES,
    threshold=0.25
)

structural_only_results = run_xgb_feature_group_experiment(
    train_full,
    test_full,
    STRUCTURAL_FEATURES,
    threshold=0.25
)

full_feature_results = run_xgb_feature_group_experiment(
    train_full,
    test_full,
    FULL_FEATURE_SET,
    threshold=0.25
)

print("Experiments completed.")

Experiments completed.


In [41]:
# Structural feature contribution summary

structural_contribution_results = pd.DataFrame([

    {
        "feature_configuration": "Behavioral + Temporal",
        "num_features": len(BEHAVIORAL_TEMPORAL_FEATURES),
        "roc_auc": round(behavioral_results["roc_auc"], 4),
        "pr_auc": round(behavioral_results["pr_auc"], 4),
        "f1": round(behavioral_results["f1"], 4),
        "precision": round(behavioral_results["precision"], 4),
        "recall": round(behavioral_results["recall"], 4),
    },

    {
        "feature_configuration": "Structural Only",
        "num_features": len(STRUCTURAL_FEATURES),
        "roc_auc": round(structural_only_results["roc_auc"], 4),
        "pr_auc": round(structural_only_results["pr_auc"], 4),
        "f1": round(structural_only_results["f1"], 4),
        "precision": round(structural_only_results["precision"], 4),
        "recall": round(structural_only_results["recall"], 4),
    },

    {
        "feature_configuration": "Full Feature Set",
        "num_features": len(FULL_FEATURE_SET),
        "roc_auc": round(full_feature_results["roc_auc"], 4),
        "pr_auc": round(full_feature_results["pr_auc"], 4),
        "f1": round(full_feature_results["f1"], 4),
        "precision": round(full_feature_results["precision"], 4),
        "recall": round(full_feature_results["recall"], 4),
    }

])

display(structural_contribution_results)

,feature_configuration,num_features,roc_auc,pr_auc,f1,precision,recall
0,Behavioral + Temporal,19,0.7068,0.4401,0.4787,0.3677,0.6855
1,Structural Only,6,0.7014,0.4311,0.4703,0.3467,0.7312
2,Full Feature Set,25,0.7064,0.4395,0.4779,0.3678,0.6822


### Interpretation of Structural Feature Contribution

The structural feature contribution analysis reveals that leakage-safe structural interaction features contain substantial predictive information for the re-engagement task.

Most notably, the structural-only feature configuration achieves a ROC-AUC of approximately 0.70 despite using only six engineered structural features. This indicates that interaction structure alone captures important behavioral regularities related to user return patterns.

At the same time, the full feature set does not outperform the behavioral and temporal configuration, with ROC-AUC remaining nearly identical across both configurations (0.7064 vs. 0.7068). This suggests that the structural features may partially overlap with information already captured by the behavioral and pair-history features.

Overall, these findings support SQ2 by demonstrating that leakage-safe structural interaction features contribute meaningful predictive signal, even though their incremental contribution beyond the broader behavioral feature set remains limited.

In [42]:
# Save structural contribution analysis

STRUCTURAL_RESULTS_PATH = (
    TABLE_DIR / "structural_feature_contribution.csv"
)

structural_contribution_results.to_csv(
    STRUCTURAL_RESULTS_PATH,
    index=False
)

print("Saved to:", STRUCTURAL_RESULTS_PATH)

Saved to: ../reports/modeling/tables/structural_feature_contribution.csv


## XGBoost Feature Importance Analysis

To complement the feature group ablation analysis, feature importance is inspected for the tuned XGBoost model trained on the full feature set.

This analysis provides a descriptive view of which engineered features are most frequently used by the tree-based model. The results should be interpreted cautiously because feature importance can be affected by correlations between features and does not provide causal evidence.

Nevertheless, it helps identify which behavioral, temporal, and structural signals are most influential in the final tabular model.

In [43]:
# Train tuned XGBoost on the full feature set for feature importance analysis

xgb_importance_model = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("model", XGBClassifier(
        max_depth=4,
        learning_rate=0.05,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.0,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

xgb_importance_model.fit(
    train_full[FULL_FEATURE_SET],
    train_full["label"]
)

print("XGBoost model trained for feature importance analysis.")

XGBoost model trained for feature importance analysis.


In [44]:
# Extract feature importance values

xgb_model = xgb_importance_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": FULL_FEATURE_SET,
    "importance": xgb_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance.head(15))

,feature,importance
0,pair_total_watch_time,0.362808
1,pair_watch_share_user,0.210156
2,pair_watch_share_streamer,0.209666
3,pair_past_sessions,0.098245
4,pair_recency,0.015425
5,time_cos_week,0.012016
6,streamer_repeat_audience_ratio,0.010489
7,user_past_sessions,0.009080
8,time_sin_week,0.008839
9,streamer_total_watch_time,0.008646


### Interpretation of Feature Importance Analysis

The feature importance analysis further supports the findings of the structural contribution experiments.

The most influential features are dominated by pair-level interaction intensity and structural relationship measures, particularly:

- `pair_total_watch_time`
- `pair_watch_share_user`
- `pair_watch_share_streamer`

This indicates that the strength and concentration of historical user–streamer interactions are highly predictive of future re-engagement behavior.

More generally, the results suggest that user re-engagement is driven less by isolated user-level or streamer-level popularity signals and more by the historical interaction dynamics between specific user–streamer pairs.

The prominence of the structural pair-share features also supports the broader conclusion that leakage-safe structural interaction information contains substantial predictive value, even when implemented through relatively simple engineered statistics rather than fully learned graph representations.

In [45]:
# Save feature importance table

FEATURE_IMPORTANCE_PATH = TABLE_DIR / "xgb_feature_importance.csv"

feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

print("Saved to:", FEATURE_IMPORTANCE_PATH)

Saved to: ../reports/modeling/tables/xgb_feature_importance.csv


## Methodological Limitations and Reflections

Several methodological limitations should be considered when interpreting the graph-based modeling results.

First, the LightGCN implementation follows a static transductive setup. The interaction graph is constructed only from training-period interactions, meaning that embeddings can only be learned for users and streamers observed during training. As a result, validation and test anchors containing unseen nodes are excluded from graph-based evaluation. Although the graph-covered test subset retained a nearly identical class distribution to the full temporal test set, this filtering still limits direct comparability between graph-based and full-population evaluations.

Second, the interaction graph itself remains relatively simple. The implemented LightGCN model uses an unweighted static bipartite graph without explicit temporal edge modeling, interaction decay, or edge weighting based on engagement intensity. Consequently, the graph representation may fail to capture important temporal dynamics underlying user re-engagement behavior.

Third, the LightGCN training procedure was computationally constrained due to CPU-based execution. The model was trained for only a limited number of epochs, and the implementation required repeated full-graph propagation during mini-batch optimization. Diagnostic analysis further revealed strong score saturation, where predicted probabilities concentrated near 1.0 for both classes. This behavior likely reduced the discriminative capacity of the graph-only predictions.

Finally, the graph-based evaluation focuses on a single graph architecture and representation strategy. Alternative approaches such as temporal graph neural networks, weighted interaction graphs, inductive graph models (e.g., GraphSAGE), or sequence-aware recommendation architectures may produce different results and remain valuable directions for future work.

## Phase 6 Conclusion

This phase compared traditional machine learning models and graph-based approaches for predicting user re-engagement on live streaming platforms under a leakage-safe temporal evaluation framework.

The results showed that Logistic Regression and tuned XGBoost achieved highly similar and stable predictive performance, reaching ROC-AUC values around 0.70 on the temporal test set. These findings indicate that carefully engineered behavioral, temporal, and structural interaction features already capture substantial predictive information for the re-engagement task.

In contrast, the graph-based LightGCN approach produced substantially weaker results in its graph-only configuration. Furthermore, combining LightGCN-derived embeddings with the tabular feature representation did not improve performance over the tabular-only baseline when evaluated on the same graph-covered test subset.

Overall, the findings suggest that, within the implemented static graph setup, graph-based representations provide limited additional predictive value beyond leakage-safe engineered behavioral features.

These results directly address the thesis research questions and establish the empirical basis for the final discussion chapter, where the findings will be interpreted in relation to prior literature, methodological trade-offs, and broader implications for re-engagement prediction in live streaming environments.